# L'économie et sa vitesse de croisière · *The economy and its cruising speed*

Notebook compagnon du chapitre **15. Croissance potentielle et output gap : la carte que personne ne peut mesurer** — [lire l'article](https://nmlab.io/ressources/croissance-potentielle-et-output-gap).
Companion notebook to chapter **15. Potential Growth and the Output Gap: The Map No One Can Measure** — [read the article](https://nmlab.io/en/ressources/potential-growth-and-output-gap).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_gdp() -> tuple[Series, Series]:
    """PIB réel (GDPC1) et PIB potentiel (GDPPOT), en direct depuis FRED.
    U.S. real GDP and potential GDP (CBO), live from FRED."""
    real = nm.load_fred("GDPC1")
    potential = nm.load_fred("GDPPOT").loc[:real.index[-1]]   # borne aux données réelles
    return real, potential

real, potential = load_gdp()


import numpy as np
import pandas as pd
import matplotlib.dates as mdates
from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="L'économie et sa vitesse de croisière",
        sub="PIB réel et PIB potentiel des États-Unis, 1949-2026",
        ylab="milliards de dollars de 2017 (échelle log)",
        real="PIB réel", potential="PIB potentiel (CBO)",
        gfc="2009", covid="COVID\n2020",
        note="Le PIB potentiel (la « ligne » soutenable) ne se mesure pas : il s'estime, et se révise. Le PIB réel oscille\n"
             "autour de lui — en dessous dans les récessions. Source : BEA et CBO via FRED (GDPC1, GDPPOT)."),
    "en": dict(
        title="The economy and its cruising speed",
        sub="U.S. real GDP and potential GDP, 1949-2026",
        ylab="billions of 2017 dollars (log scale)",
        real="real GDP", potential="potential GDP (CBO)",
        gfc="2009", covid="COVID\n2020",
        note="Potential GDP (the sustainable « line ») is not measured: it is estimated, and revised. Real GDP oscillates\n"
             "around it — below it in recessions. Source: BEA and CBO via FRED (GDPC1, GDPPOT)."),
}

_SUP = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")

def _sci(value: float, _) -> str:
    """Formate un tick logarithmique en « n × 10ᵏ » (exposant Unicode)."""
    exp = int(np.floor(np.log10(value)))
    mant = int(round(value / 10 ** exp))
    return f"{mant} × 10{str(exp).translate(_SUP)}"

def build_figure(real: Series, potential: Series, lang: str) -> Figure:
    """PIB réel (bleu plein) et potentiel (ambre pointillé) en échelle log."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1064)
    ax = nm.axes(fig, left=0.092)
    h_real, = ax.plot(real.index, real, color=nm.COLORS["blue"], linewidth=2.9,
                      label=text["real"], zorder=2)
    h_pot, = ax.plot(potential.index, potential, color=nm.COLORS["amber"], linewidth=3.0,
                     linestyle=(0, (6, 4)), label=text["potential"], zorder=3)
    ax.set_yscale("log")
    ax.set_ylim(2000, 33000)
    ax.set_yticks([2000, 5000, 10000, 20000])
    ax.yaxis.set_major_formatter(nm.thousands(lang))
    ax.set_yticks([3000, 4000, 6000, 30000], minor=True)
    ax.yaxis.set_minor_formatter(FuncFormatter(_sci))
    ax.tick_params(axis="y", which="minor", labelsize=20, labelcolor=nm.COLORS["muted"])
    ax.grid(which="minor", visible=False)
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend([h_pot, h_real], [text["potential"], text["real"]], loc="upper left",
              frameon=False, fontsize=23, labelcolor=nm.COLORS["text"],
              handlelength=2.6, borderaxespad=1.4)
    # Annotations : creux de 2009 et choc COVID de 2020.
    ax.annotate(text["gfc"], xy=(pd.Timestamp("2009-01-01"), float(real.loc["2009"].min())),
                xytext=(pd.Timestamp("2011-06-01"), 8800), ha="center", va="center",
                fontsize=22, color=nm.COLORS["muted"],
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["muted"], lw=1.6))
    ax.annotate(text["covid"], xy=(pd.Timestamp("2020-04-01"), float(real.loc["2020"].min())),
                xytext=(pd.Timestamp("2015-06-01"), 24200), ha="center", va="center",
                fontsize=22, color=nm.COLORS["muted"], linespacing=1.4,
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["muted"], lw=1.6))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(real, potential, LANG)